# YOLO26s + LoRA 独立冒烟验证

仅验证 CUDA、LoRA 梯度、Adapter 保存、权重合并、固定 test 评估和 ONNX 导出。输出位于 `models/trained/yolo26_lora/smoke/`，不得部署。

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / 'training').is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Project:', PROJECT_ROOT)

In [ ]:
import torch, ultralytics
from training.train_yolo26_lora import BASE_MODEL, train

print('Python:', sys.version.split()[0])
print('Ultralytics:', ultralytics.__version__, ultralytics.__file__)
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Protected baseline:', BASE_MODEL)
if '.venv-yolo26' not in str(Path(ultralytics.__file__).resolve()):
    raise RuntimeError('请切换到项目独立 YOLO26 Kernel')
if not torch.cuda.is_available():
    raise RuntimeError('LoRA 冒烟验证要求 CUDA')

In [ ]:
best = train(
    smoke=True, epochs=3, batch=4, imgsz=640,
    rank=8, alpha=16.0, dropout=0.05,
    run_name='yolo26s_lora_smoke_manual',
    resume=False, export_onnx=True,
)
print('SMOKE ONLY merged best:', best)

In [ ]:
import json
manifest = json.loads((best.parents[1] / 'lora_manifest.json').read_text(encoding='utf-8'))
manifest